# Context & Dependency Injection

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) OpenAI Agents SDK roadmap.

You will learn how to use `RunContextWrapper` for dependency injection, pass shared state to tools, and use `ToolContext` for tool metadata.

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Define a Context Type

Create a dataclass to hold shared state that tools and agents will need.

In [ ]:
from dataclasses import dataclass


@dataclass
class UserContext:
    user_id: str
    user_name: str
    is_premium: bool = False

## 4. Tools Receive Context Automatically

When you type-hint the first argument as `RunContextWrapper[T]`, the SDK injects the context object automatically. The model only sees the remaining parameters.

In [ ]:
from agents import Agent, Runner, function_tool, RunContextWrapper


@function_tool
def get_user_profile(ctx: RunContextWrapper[UserContext]) -> str:
    """Get the current user's profile information."""
    user = ctx.context
    status = "Premium" if user.is_premium else "Free"
    return f"User: {user.user_name} (ID: {user.user_id}), Plan: {status}"


agent = Agent(
    name="Profile Agent",
    instructions="You help users view their profile. Use the get_user_profile tool.",
    tools=[get_user_profile],
)

ctx = UserContext(user_id="u_123", user_name="Alice", is_premium=True)
result = await Runner.run(agent, "Show me my profile.", context=ctx)
print(result.final_output)

## 5. Context with Regular Parameters

Tools can accept both context and model-facing parameters. Context is always the first argument.

In [ ]:
@function_tool
def check_order(ctx: RunContextWrapper[UserContext], order_id: str) -> str:
    """Check the status of an order for the current user."""
    return f"Order {order_id} for user {ctx.context.user_name}: Shipped"


order_agent = Agent(
    name="Order Agent",
    instructions="You help users check their orders.",
    tools=[check_order],
)

ctx = UserContext(user_id="u_123", user_name="Alice")
result = await Runner.run(order_agent, "Check order ORD-456.", context=ctx)
print(result.final_output)

## 6. ToolContext for Metadata

`ToolContext` extends `RunContextWrapper` with `tool_name` and `tool_call_id` — useful for logging and auditing.

In [ ]:
from agents import ToolContext


@function_tool
def log_action(ctx: ToolContext[UserContext], action: str) -> str:
    """Log a user action with metadata."""
    return (
        f"Logged: {action} by {ctx.context.user_name} "
        f"(tool: {ctx.tool_name}, call_id: {ctx.tool_call_id})"
    )


log_agent = Agent(
    name="Logger",
    instructions="Log the user's actions using the log_action tool.",
    tools=[log_action],
)

ctx = UserContext(user_id="u_123", user_name="Alice")
result = await Runner.run(log_agent, "Log that I viewed my dashboard.", context=ctx)
print(result.final_output)

## 7. Dynamic Instructions with Context

Agent instructions can be a function that receives the context, enabling personalized behavior at runtime.

In [ ]:
def dynamic_instructions(ctx: RunContextWrapper[UserContext], agent: Agent) -> str:
    user = ctx.context
    tier = "premium" if user.is_premium else "free"
    return f"You are helping {user.user_name}, a {tier} user. Be extra helpful to premium users."


support_agent = Agent(
    name="Support Agent",
    instructions=dynamic_instructions,
)

ctx = UserContext(user_id="u_123", user_name="Alice", is_premium=True)
result = await Runner.run(support_agent, "I need help with my account.", context=ctx)
print(result.final_output)

## Key Takeaways

- Define a context type (dataclass) to hold shared state like user info or config
- Pass context to `Runner.run(context=obj)` — it flows through the entire run
- Tools receive context as their first argument via `RunContextWrapper[T]`
- `ToolContext[T]` extends the wrapper with `tool_name` and `tool_call_id` metadata
- Agent instructions can be a function that receives context for dynamic personalization